# Notebook Overview
Purpose:

*   Transform Bronze customer data into a standardized Silver dataset.
*   Validate customer records against documented data-quality rules.
*   Separate accepted and quarantined records.
*   Persist outputs and reconcile quality metrics.

### Configuration / Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"
BRONZE_CUSTOMER_PATH = f"{FILE_PATH}/bronze/customers"
QUARANTINE_CUSTOMER_PATH = f"{FILE_PATH}/quarantine"
SILVER_PATH = f"{FILE_PATH}/silver"

Mounted at /content/drive


### Spark Session

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName('customer-silver').getOrCreate()
spark.version

'4.0.4'

### Load Data from Bronze

In [5]:
def load_data(FILE_PATH):
  try:
    df_customer = spark.read.parquet(FILE_PATH).withColumn(
        '_source_file', F.lit(F.input_file_name())

    )
    print('Data Load Successful')
    return df_customer

  except Exception as e:
    raise RuntimeError(f"Data Load Failed: {str(e)}")

df_customer = load_data(BRONZE_CUSTOMER_PATH)

Data Load Successful


### Row Count and Bronze Customer Schema

In [6]:
def schema_and_count(df):
  print('Bronze Customer Row Count:', df.count())
  print('Bronze Customer Schema')
  print('----------------------')
  df.printSchema()

### Validate the Bronze Schema

In [7]:
def validate_customers_schema(df):
  try:
    expected_columns = ['customer_id',
                        'first_name',
                        'last_name',
                        'email',
                        'country',
                        'city',
                        'signup_date',
                        'customer_segment',
                        'account_status',
                        '_ingestion_timestamp',
                        '_source_file']

    df_column = df.columns

    missing_cols = list(set(expected_columns) - set(df_column))

    expected_schema = {
          'customer_id': 'string',
          'first_name': 'string',
          'last_name': 'string',
          'email': 'string',
          'country': 'string',
          'city': 'string',
          'signup_date': 'string',
          'customer_segment': 'string',
          'account_status': 'string',
          '_ingestion_timestamp': 'timestamp',
          '_source_file': 'string'
      }

    actual_types = dict(df.dtypes)

    type_missmatch = []
    for col, expected_type in expected_schema.items():
      if col in actual_types and actual_types[col] != expected_type:
        type_missmatch.append(f"Column '{col}' expected '{expected_type}', but got '{actual_types[col]}'")

    error_messages = []
    if missing_cols:
      error_messages.append(f'Missing_cols: {missing_cols}')
    if type_missmatch:
      error_messages.append(f'Type Missmatch: {type_missmatch}')

    if error_messages:
      raise ValueError("Customer Schema Validation Failed:\n" + "\n".join(error_messages))
    print("Schema Validation passed successfully!")

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_customers_schema': {str(e)}") from e

### Preserve Audit Information

In [8]:
from pyspark.sql import functions as F
def preserve_audit(df):
  df_preserve = df
  df_preserve = df.withColumn(f'_original_signup_date', F.col('signup_date'))
  return df_preserve

### Standardize String Columns

In [9]:
def standardize_customer_strings(df):
  try:
    string_cols = [
        'customer_id',
        'first_name',
        'last_name',
        'email',
        'country',
        'city',
        'signup_date',
        'customer_segment',
        'account_status',
        '_source_file'
    ]

    df_clean = df

    for c in string_cols:
      df_clean = df_clean.withColumn(
          c,
          F.when(F.trim(F.lower(F.col(c))).isin('n/a', 'na', 'unknown', 'null', ''), None).otherwise(F.trim(F.col(c)))
      )

    if 'email' in df_clean.columns:
      df_clean = df_clean.withColumn('email', F.lower(F.col('email')))

    return df_clean
  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'standardize_customer_strings': {str(e)}") from e

### Cast Types

In [10]:
from pyspark.sql.types import StringType, DateType

def cast_customer_types(df):
  try:
    if 'signup_date' in df.columns:
      df_typed = df.withColumn('signup_date', F.col('signup_date').try_cast(DateType()))

    if 'customer_id' in df.columns:
      df_typed = df_typed.withColumn('customer_id', F.col('customer_id').cast('string'))

    return df_typed

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'cast_customer_types': {str(e)}") from e


### Validate Customer Records

In [11]:
def validate_customer_records(df):
  try:
    approved_segments = ['consumer', 'premier', 'enterprise', 'small business']
    approved_status = ['suspended', 'active', 'dormant', 'inactive']

    df_validated = df.withColumn(
        'is_valid',
        F.when(F.col('customer_id').isNull(), 'Missing or Null Customer ID')
        .when((F.col('signup_date').isNull()) & (F.col('_original_signup_date').isNotNull()), 'Invalid Date Format')
        .when((F.col('email').isNull()) | ((~F.col('email').rlike(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')))
        ,'Invalid Email Format')
        .when((F.col('customer_segment').isNull()) | (~F.lower(F.col('customer_segment')).isin(approved_segments)), 'Invalid Customer Segment')
        .when((F.col('account_status').isNull()) | (~F.lower(F.col('account_status')).isin(approved_status)), 'Invalid Account Status')
        .otherwise(None)
    ).withColumn('country', F.when(F.col('city').isin('Brisbane', 'Sydney', 'Perth', 'Melbourne'), 'Australia').otherwise(F.col('country')))

    return df_validated

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_customer_records': {str(e)}") from e

### Handle Duplicate Customer IDs

In [12]:
from pyspark.sql import Window as W

def handle_duplicate_customer_ids(df):
  try:
    window_spec = W.partitionBy('customer_id').orderBy(F.col('_ingestion_timestamp').desc())

    df_ranked = df.withColumn(
        'rank',
        F.row_number().over(window_spec)
    )

    df_mark = df_ranked.withColumn(
        'is_valid',
        F.coalesce(
            F.col('is_valid'),
            F.when(F.col('rank') > 1, 'Duplicate Customer ID')
        )
    ).drop('rank')

    return df_mark

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'handle_duplicate_customer_ids': {str(e)}") from e

### Split Valid and Quarantined Records

In [13]:
def split_customers_by_quality(df):
  try:
    df_valid = df.filter(F.col('is_valid').isNull()).drop('is_valid', '_original_signup_date')

    df_quarantine = df.filter(F.col('is_valid').isNotNull())

    return df_valid, df_quarantine

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'split_customers_by_quality': {str(e)}") from e

### Orchestrate the Customer Transformation

In [14]:
def transform_customers(df):
  try:
    print('===============================')
    print('TRANSFORMING DATA')
    print('===============================')
    print('Start: Validating Bronze Schema')
    validate_customers_schema(df)
    print('Finished: Validaing Bronze Schema')
    print('Start: Preserving Data for Audit')
    preserved_df = preserve_audit(df)
    print('Finished: Preserving Data for Audit')
    print('Start: Standardizing String Columns')
    standardize_df = standardize_customer_strings(preserved_df)
    print('Finished: Standardizing String Columns')
    print('Start: Cast type Column')
    casted_df = cast_customer_types(standardize_df)
    print('Finished: Cast type Column')
    print('Start: Validating Customer Record')
    validated_df = validate_customer_records(casted_df)
    print('Finished: Validating Customer Record')
    print('Start: Handling and Marking duplicate id')
    df_mark = handle_duplicate_customer_ids(validated_df)
    print('Finished: Handling and Marking duplicate id')
    print('Spiliting and Returning Validated and Quanrantine data')
    print('===============================')
    print('TRANSFORMING COMPLETE')
    print('===============================')
    return split_customers_by_quality(df_mark)

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Failed inside 'transform_customers': {str(e)}") from e

### **Transforming Customer Data**

In [15]:
valid_df, quarantine_df = transform_customers(df_customer)

TRANSFORMING DATA
Start: Validating Bronze Schema
Schema Validation passed successfully!
Finished: Validaing Bronze Schema
Start: Preserving Data for Audit
Finished: Preserving Data for Audit
Start: Standardizing String Columns
Finished: Standardizing String Columns
Start: Cast type Column
Finished: Cast type Column
Start: Validating Customer Record
Finished: Validating Customer Record
Start: Handling and Marking duplicate id
Finished: Handling and Marking duplicate id
Spiliting and Returning Validated and Quanrantine data
TRANSFORMING COMPLETE


### Calculate quality metrics and reconcile

In [16]:
def generate_pipeline_metrics(df_bronze, df_valid, df_quarantine):
    print("\n" + "=" * 65)
    print(" PIPELINE RECONCILIATION & QUALITY METRICS REPORT")
    print("=" * 65)

    input_count = df_bronze.count()
    silver_count = df_valid.count()
    quarantine_count = df_quarantine.count()

    print("\n[1] RECONCILIATION SUMMARY")
    print(f"  • Total Bronze Input Count      : {input_count:,}")
    print(f"  • Accepted Silver Count         : {silver_count:,}")
    print(f"  • Quarantined Count             : {quarantine_count:,}")

    sum_outputs = silver_count + quarantine_count
    is_balanced = (input_count == sum_outputs)
    print(f"  • Reconciliation Balance Check  : {'PASSED (Match)' if is_balanced else 'FAILED (Mismatch!)'}")
    if not is_balanced:
        print(f"    WARNING: Input ({input_count:,}) != Silver ({silver_count:,}) + Quarantine ({quarantine_count:,})")

    print("\n[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)")
    if quarantine_count > 0:
        print("  Breakdown of reasons records were quarantined:")
        df_quarantine.groupBy("is_valid").count().orderBy(F.col("count").desc()).show(truncate=False)
    else:
        print("  • No quarantined records found. 100% clean dataset!")

    print("\n[3] NULL COUNTS IN SILVER LAYER")
    print("  (Tracking optional profile fields alongside required ones)")
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_valid.columns]
    df_valid.select(null_exprs).show(truncate=False)

    print("\n[4] UNIQUENESS CHECK (customer_id in Silver)")
    distinct_customers = df_valid.select("customer_id").distinct().count()
    is_unique = (silver_count == distinct_customers)
    print(f"  • Total Silver Rows             : {silver_count:,}")
    print(f"  • Unique Customer IDs           : {distinct_customers:,}")
    print(f"  • Uniqueness Constraint         : {'PASSED (No duplicate IDs in Silver)' if is_unique else 'FAILED (Duplicates present!)'}")

    print("=" * 65)

generate_pipeline_metrics(df_customer, valid_df, quarantine_df)


 PIPELINE RECONCILIATION & QUALITY METRICS REPORT

[1] RECONCILIATION SUMMARY
  • Total Bronze Input Count      : 10,000
  • Accepted Silver Count         : 9,997
  • Quarantined Count             : 3
  • Reconciliation Balance Check  : PASSED (Match)

[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)
  Breakdown of reasons records were quarantined:
+---------------------+-----+
|is_valid             |count|
+---------------------+-----+
|Invalid Email Format |1    |
|Invalid Date Format  |1    |
|Duplicate Customer ID|1    |
+---------------------+-----+


[3] NULL COUNTS IN SILVER LAYER
  (Tracking optional profile fields alongside required ones)
+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+--------------------+------------+
|customer_id|first_name|last_name|email|country|city|signup_date|customer_segment|account_status|_ingestion_timestamp|_source_file|
+-----------+----------+---------+-----+-------+----+-----------+----

### Adding Metdata and Enforcing Schema

In [18]:
from pyspark.sql.types import StructType, StructField, TimestampType


# Adding metadata to Quarantine Dataset
quarantine_df = quarantine_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

# Adding metadata to Valid Dataset
valid_df = valid_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

silver_customer_df = valid_df.select(
    'customer_id',
    'first_name',
    'last_name',
    'email',
    'country',
    'city',
    'signup_date',
    'customer_segment',
    'account_status',
    '_silver_processed_timestamp',
    '_source_file'
)

silver_customer_df.printSchema()
silver_customer_df.show(5)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = true)

+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+---------------------------+--------------------+
|customer_id|first_name|last_name|               email|      country|   city|signup_date|customer_segment|account_status|_silver_processed_timestamp|        _source_file|
+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+---------------------------+--------------------+
|

### ***Write into Silver***

In [24]:
try:
  print('INFO: Writing records to Quarantine layer...')
  # Write Quarantine Data
  quarantine_df.write.mode('overwrite').parquet(f"{QUARANTINE_CUSTOMER_PATH}/customers")
  print('SUCCESS: Quarantine dataset written successfully.')

  print('INFO: Writing clean records to Silver layer...')
  # Write Siver Data into Layer
  silver_customer_df.write.mode('overwrite').parquet(f"{SILVER_PATH}/customers")
  print('SUCCESS: Silver dataset written successfully.')

except Exception as e:
    raise RuntimeError(f"[Pipeline Error] Failed to write Silver dataset to '{SILVER_PATH}/customers': {str(e)}") from e


INFO: Writing records to Quarantine layer...
SUCCESS: Quarantine dataset written successfully.
INFO: Writing clean records to Silver layer...
SUCCESS: Silver dataset written successfully.


### Read Back and Verify

In [22]:
customer_silver = spark.read.parquet(f"{SILVER_PATH}/customers")

customer_silver.show(5)

customer_silver.count()

+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+---------------------------+--------------------+
|customer_id|first_name|last_name|               email|      country|   city|signup_date|customer_segment|account_status|_silver_processed_timestamp|        _source_file|
+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+---------------------------+--------------------+
|    C000001|       Mia|    Davis|mia.davis1@exampl...|      Germany| Munich| 2024-10-14|  Small Business|        Active|       2026-10-20 00:53:...|file:///content/d...|
|    C000002|     Arjun|    Davis|arjun.davis2@exam...|       Canada|Calgary| 2025-08-11|        Consumer|        Active|       2026-10-20 00:53:...|file:///content/d...|
|    C000003|      Noah|   Wilson|noah.wilson3@exam...|       Canada|Calgary| 2021-02-24|        Consumer|        Active|       2026-10-20 00:53:

9997